# Task 9 — HYDE Basin Aggregation and s/u Characterization

**Goal**: Characterize how HYDE 3.4 land-use variables (cropland, grazing_land) aggregate to L8 and L6 basin polygons, 
and how the local/upstream (s/u) duality behaves for these variables compared to the static climate divergences 
characterized in Task 3.

**Key questions**:
1. Do centroid lookup and polygon-interior aggregation agree? Under what conditions do they diverge?
2. How wide are HYDE s/u divergences compared to Task 3 climate divergences? Is the 'downstream-of-civilization' prediction borne out?
3. Does HYDE 2000 CE basin-aggregated cropland agree with static BasinATLAS `crp_pc_sse`/`crp_pc_use`?
4. Do results differ meaningfully between L8 (sub-basin) and L6 (major basin) scales?

**Levels**: L8 — 500-basin stratified sample (25/cluster); L6 — full population (16,397 basins)  
**Variables**: cropland, grazing_land  
**Epochs**: -1000 BCE, 0 CE, 1000 CE, 2000 CE  
**Aggregation**: polygon-interior for `s` (query basin); centroid for upstream basins feeding `u`

In [3]:
# Cell 1 — Imports and config
import sys, time
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import shapely
from shapely.geometry import Point
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

sys.path.insert(0, '../../..')
from scripts.shared.db_utils import db_connect

# Paths
HYDE_DIR = '../../../data/hyde/NetCDF/'
CLUSTER_FILE = '../../../output/edop/explore/05_cluster_assignments.csv'
OUT_DIR = '../../../output/edop/explore/'

# Target epochs (HYDE time axis uses astronomical year numbering — 0 = 1 BCE in HYDE)
TARGET_YEARS = [-1000, 0, 1000, 2000]
EPOCH_LABELS = {-1000: '1000 BCE', 0: '1 CE', 1000: '1000 CE', 2000: '2000 CE'}

HYDE_VARS = ['cropland', 'grazing_land']

print('Imports OK')
print('shapely version:', shapely.__version__)

Imports OK
shapely version: 2.1.2


## 1. Load basin topology from DB

In [4]:
# Cell 2 — Load basin topology (L8 + L6)
conn = db_connect()

# Load all L8 topology (lightweight — no geometry yet)
t0 = time.time()
topo8 = pd.read_sql("""
    SELECT hybas_id, next_down, sub_area, endo,
           ST_X(ST_Centroid(geom)) AS lon,
           ST_Y(ST_Centroid(geom)) AS lat,
           crp_pc_sse, crp_pc_use
    FROM public.basin08
""", conn)
print(f'L8 topology loaded: {len(topo8):,} rows in {time.time()-t0:.1f}s')

# Load all L6 topology
t0 = time.time()
topo6 = pd.read_sql("""
    SELECT hybas_id, next_down, sub_area, endo,
           ST_X(ST_Centroid(geom)) AS lon,
           ST_Y(ST_Centroid(geom)) AS lat,
           crp_pc_sse, crp_pc_use
    FROM public.basin06
""", conn)
print(f'L6 topology loaded: {len(topo6):,} rows in {time.time()-t0:.1f}s')

conn.close()

# Set hybas_id as index for fast lookup
topo8 = topo8.set_index('hybas_id')
topo6 = topo6.set_index('hybas_id')

print(f'\nL8 sub_area: median={topo8.sub_area.median():.0f} km², max={topo8.sub_area.max():.0f} km²')
print(f'L6 sub_area: median={topo6.sub_area.median():.0f} km², max={topo6.sub_area.max():.0f} km²')

L6 topology loaded: 16,397 rows in 1.4s

L8 sub_area: median=476 km², max=115570 km²
L6 sub_area: median=5318 km², max=217174 km²


## 2. Draw L8 and L6 samples (25/cluster)

In [5]:
# Cell 3 — Draw stratified samples
clusters8 = pd.read_csv(CLUSTER_FILE)
CLUSTER_FILE_L6 = '../../../output/edop/explore/05_cluster_assignments_L6.csv'
clusters6 = pd.read_csv(CLUSTER_FILE_L6)

print('L8 cluster sizes:\n', clusters8['cluster_id'].value_counts().sort_index())
print('\nL6 cluster sizes:\n', clusters6['cluster_id'].value_counts().sort_index())

# Stratified sample: 25 per cluster, both levels
np.random.seed(42)

sample8_ids = (
    clusters8.groupby('cluster_id')
    .apply(lambda g: g.sample(min(25, len(g)), random_state=42))
    .reset_index(drop=True)
    ['hybas_id'].astype(float).values
)

sample6_ids = (
    clusters6.groupby('cluster_id')
    .apply(lambda g: g.sample(min(25, len(g)), random_state=42))
    .reset_index(drop=True)
    ['hybas_id'].astype(float).values
)

print(f'\nL8 sample: {len(sample8_ids)} basins (25/cluster × 20 clusters)')
print(f'L6 sample: {len(sample6_ids)} basins (25/cluster × 20 clusters)')

/var/folders/w1/ms_2x6rj0ls88v79q33lvds80000gp/T/ipykernel_67451/3948315847.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(25, len(g)), random_state=42))
/var/folders/w1/ms_2x6rj0ls88v79q33lvds80000gp/T/ipykernel_67451/3948315847.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(25, len(g)), random_state=42))


## 3. Fetch basin polygons via geopandas

In [14]:
# Cell 4 — Fetch basin polygons via geopandas (sample only)
conn = db_connect()

# L8 sample polygons (500 basins)
ids8_str = ','.join(str(int(x)) for x in sample8_ids)
t0 = time.time()
geom8 = gpd.read_postgis(
    f"SELECT hybas_id, geom FROM public.basin08 WHERE hybas_id IN ({ids8_str})",
    conn, geom_col='geom', index_col='hybas_id'
).rename_geometry('geometry')
print(f'L8 sample polygons: {len(geom8)} rows in {time.time()-t0:.1f}s')

# L6 sample polygons (500 basins)
ids6_str = ','.join(str(int(x)) for x in sample6_ids)
t0 = time.time()
geom6 = gpd.read_postgis(
    f"SELECT hybas_id, geom FROM public.basin06 WHERE hybas_id IN ({ids6_str})",
    conn, geom_col='geom', index_col='hybas_id'
).rename_geometry('geometry')
print(f'L6 sample polygons: {len(geom6)} rows in {time.time()-t0:.1f}s')

conn.close()

L6 sample polygons: 500 rows in 0.2s


## 4. Build upstream DAGs in Python

In [10]:
# Cell 5 — Build upstream DAGs
def build_upstream_dag(topo_df):
    """Build dict: next_down_id -> [upstream hybas_ids]. Excludes endorheic basins."""
    mask = (topo_df['endo'] == 0) & (topo_df['next_down'] > 0)
    filtered = topo_df[mask][['next_down']].copy()
    filtered.index.name = 'hid'
    filtered = filtered.reset_index()
    return filtered.groupby('next_down')['hid'].apply(list).to_dict()

def get_all_upstream(hid, dag):
    """BFS: return list of all upstream hybas_ids (not including hid itself)."""
    visited = []
    queue = dag.get(hid, [])[:]
    while queue:
        node = queue.pop()
        visited.append(node)
        queue.extend(dag.get(node, []))
    return visited

t0 = time.time()
dag8 = build_upstream_dag(topo8)
print(f'L8 DAG built: {len(dag8):,} nodes with upstream children in {time.time()-t0:.1f}s')

t0 = time.time()
dag6 = build_upstream_dag(topo6)
print(f'L6 DAG built: {len(dag6):,} nodes with upstream children in {time.time()-t0:.1f}s')

# Sanity check: Timbuktu upstream area should be ~379,818 km²
tim_id = 1080561810.0
if tim_id in topo8.index:
    up = get_all_upstream(tim_id, dag8)
    up_area = topo8.loc[topo8.index.isin(up), 'sub_area'].sum()
    print(f'\nTimbuktu upstream basins: {len(up)}, total area: {up_area:.0f} km² (expect ~379,818 km²)')

L8 DAG built: 66,768 nodes with upstream children in 0.6s
L6 DAG built: 4,679 nodes with upstream children in 0.0s

Timbuktu upstream basins: 600, total area: 379652 km² (expect ~379,818 km²)


## 5. Load HYDE arrays for target epochs

In [11]:
# Cell 6 — Load HYDE arrays for target epochs
def load_hyde_epochs(var_name, target_years):
    """
    Load HYDE variable at target years.
    Returns dict: year -> 2D numpy array (lat x lon), plus lat/lon coordinate arrays.
    """
    ds = xr.open_dataset(HYDE_DIR + f'{var_name}.nc')
    
    # Extract coordinate arrays
    lons = ds.lon.values.astype(float)
    lats = ds.lat.values.astype(float)
    
    print(f'{var_name}: lat [{lats[0]:.4f} ... {lats[-1]:.4f}], lon [{lons[0]:.4f} ... {lons[-1]:.4f}]')
    print(f'  lat ascending: {lats[0] < lats[-1]}')
    
    time_years = [t.year for t in ds.time.values]
    
    arrays = {}
    for yr in target_years:
        if yr not in time_years:
            # Find nearest
            nearest = min(time_years, key=lambda y: abs(y - yr))
            print(f'  Warning: year {yr} not in time axis, using {nearest}')
            yr_key = nearest
        else:
            yr_key = yr
        idx = time_years.index(yr_key)
        arr = ds[var_name].isel(time=idx).values.astype(float)
        arr = np.where(arr < 0, np.nan, arr)  # mask fill values
        arrays[yr] = arr
        pct_nonzero = np.nanmean(arr > 0) * 100
        print(f'  year {yr}: shape={arr.shape}, non-zero={pct_nonzero:.1f}%, max={np.nanmax(arr):.2f}')
    
    ds.close()
    return arrays, lats, lons

hyde_data = {}
hyde_lats = hyde_lons = None

for vname in HYDE_VARS:
    arrays, lats, lons = load_hyde_epochs(vname, TARGET_YEARS)
    hyde_data[vname] = arrays
    if hyde_lats is None:
        hyde_lats, hyde_lons = lats, lons

# HYDE cell size in degrees
cell_deg = abs(lats[1] - lats[0])
print(f'\nHYDE cell size: {cell_deg:.6f} deg = {cell_deg * 60:.2f} arcmin')

  year 1000: shape=(2160, 4320), non-zero=9.4%, max=85.82
  year 2000: shape=(2160, 4320), non-zero=14.0%, max=85.84

HYDE cell size: 0.083328 deg = 5.00 arcmin


## 6. Aggregation helper functions

In [15]:
# Cell 7 — Aggregation helper functions
def centroid_lookup(lon, lat, hyde_array, lons, lats):
    """Return HYDE value at the cell nearest to (lon, lat)."""
    ci = np.argmin(np.abs(lons - lon))
    cj = np.argmin(np.abs(lats - lat))
    val = hyde_array[cj, ci]
    return float(val) if not np.isnan(val) else 0.0


def polygon_interior_mean(polygon, hyde_array, lons, lats):
    """
    Mean HYDE value across all cell centers inside the polygon.
    Falls back to centroid cell if no cell center falls inside.
    Uses shapely 2.0 vectorized contains.
    Works correctly for both ascending and descending lat arrays.
    """
    minx, miny, maxx, maxy = polygon.bounds
    cell = abs(lons[1] - lons[0])  # cell size in degrees

    lon_mask = (lons >= minx - cell) & (lons <= maxx + cell)
    lat_mask = (lats >= miny - cell) & (lats <= maxy + cell)
    lon_idx = np.where(lon_mask)[0]
    lat_idx = np.where(lat_mask)[0]

    if len(lon_idx) == 0 or len(lat_idx) == 0:
        c = polygon.centroid
        return centroid_lookup(c.x, c.y, hyde_array, lons, lats), 0

    lon_sub = lons[lon_idx]
    lat_sub = lats[lat_idx]
    lon_grid, lat_grid = np.meshgrid(lon_sub, lat_sub)

    pts = shapely.points(lon_grid.ravel(), lat_grid.ravel())
    inside = shapely.contains(polygon, pts)

    if not inside.any():
        c = polygon.centroid
        return centroid_lookup(c.x, c.y, hyde_array, lons, lats), 0

    inside_lons = lon_grid.ravel()[inside]
    inside_lats = lat_grid.ravel()[inside]

    # Direct index calculation — works for ascending or descending lat/lon
    ci = np.argmin(np.abs(lons[None, :] - inside_lons[:, None]), axis=1)
    cj = np.argmin(np.abs(lats[None, :] - inside_lats[:, None]), axis=1)

    vals = hyde_array[cj, ci]
    vals = vals[~np.isnan(vals)]

    if len(vals) == 0:
        c = polygon.centroid
        return centroid_lookup(c.x, c.y, hyde_array, lons, lats), 0

    return float(vals.mean()), len(vals)


def batch_centroid_lookup(lons_arr, lats_arr, hyde_array, lons, lats):
    """Vectorized centroid lookup for arrays of lon/lat."""
    ci = np.argmin(np.abs(lons[None, :] - lons_arr[:, None]), axis=1)
    cj = np.argmin(np.abs(lats[None, :] - lats_arr[:, None]), axis=1)
    vals = hyde_array[cj, ci]
    return np.where(vals < 0, 0.0, vals)

print('Helper functions defined.')
print(f'lat direction: {"descending" if hyde_lats[0] > hyde_lats[-1] else "ascending"} — argmin indexing used throughout')

Helper functions defined.
lat direction: descending — argmin indexing used throughout


## 7. Aggregation comparison: centroid vs polygon-interior (L8 sample, 2000 CE)

In [16]:
# Cell 8 — Aggregation comparison — centroid vs polygon-interior (L8, 2000 CE)
arr_2000 = hyde_data['cropland'][2000]

comparison_rows = []
t0 = time.time()

for i, hid in enumerate(sample8_ids):
    if hid not in geom8.index:
        continue
    poly = geom8.loc[hid, 'geometry']
    trow = topo8.loc[hid]
    
    centroid_val = centroid_lookup(trow['lon'], trow['lat'], arr_2000, hyde_lons, hyde_lats)
    poly_val, n_cells = polygon_interior_mean(poly, arr_2000, hyde_lons, hyde_lats)
    
    comparison_rows.append({
        'hybas_id': hid,
        'sub_area': trow['sub_area'],
        'centroid_val': centroid_val,
        'poly_val': poly_val,
        'n_cells': n_cells,
        'diff': poly_val - centroid_val,
        'rel_diff': abs(poly_val - centroid_val) / (poly_val + 0.001)
    })
    
    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        print(f'  {i+1}/500 in {elapsed:.1f}s (est. total: {elapsed / (i+1) * 500:.0f}s)')

comp_df = pd.DataFrame(comparison_rows)
print(f'\nDone in {time.time()-t0:.1f}s')
print(f'Cells per basin: median={comp_df.n_cells.median():.0f}, p95={comp_df.n_cells.quantile(0.95):.0f}')
print(f'Centroid fallbacks (n_cells=0): {(comp_df.n_cells == 0).sum()}')
print(f'\nAbsolute diff (km²): median={comp_df["diff"].abs().median():.3f}, p95={comp_df["diff"].abs().quantile(0.95):.3f}')
print(f'Relative diff: median={comp_df.rel_diff.median():.3f}, p95={comp_df.rel_diff.quantile(0.95):.3f}')

  100/500 in 0.0s (est. total: 0s)
  200/500 in 0.1s (est. total: 0s)
  300/500 in 0.1s (est. total: 0s)
  400/500 in 0.2s (est. total: 0s)
  500/500 in 0.2s (est. total: 0s)

Done in 0.2s
Cells per basin: median=8, p95=39
Centroid fallbacks (n_cells=0): 39

Absolute diff (km²): median=0.000, p95=8.728
Relative diff: median=0.000, p95=1.000


In [17]:
# Cell 9 — Plot aggregation comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Scatter: centroid vs polygon-interior
ax = axes[0]
nonzero = comp_df[(comp_df.centroid_val > 0) | (comp_df.poly_val > 0)]
ax.scatter(nonzero.centroid_val, nonzero.poly_val, alpha=0.4, s=10)
lim = max(nonzero.centroid_val.max(), nonzero.poly_val.max())
ax.plot([0, lim], [0, lim], 'r--', lw=1)
ax.set_xlabel('Centroid lookup (km²)')
ax.set_ylabel('Polygon-interior mean (km²)')
ax.set_title('Cropland 2000 CE: centroid vs poly-interior')

# Absolute difference vs basin size
ax = axes[1]
ax.scatter(comp_df.sub_area, comp_df['diff'].abs(), alpha=0.3, s=10)
ax.set_xlabel('Basin sub_area (km²)')
ax.set_ylabel('|poly - centroid| (km²)')
ax.set_title('Disagreement vs basin size')
ax.set_xscale('log')

# Distribution of n_cells
ax = axes[2]
ax.hist(comp_df.n_cells, bins=30, edgecolor='white')
ax.set_xlabel('HYDE cells inside polygon')
ax.set_ylabel('Basin count')
ax.set_title('Cells per L8 basin (polygon-interior)')

plt.tight_layout()
plt.savefig(OUT_DIR + '09_aggregation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 09_aggregation_comparison.png')

comp_df.to_csv(OUT_DIR + '09_aggregation_comparison_L8.csv', index=False)
print('Saved 09_aggregation_comparison_L8.csv')

Saved 09_aggregation_comparison.png
Saved 09_aggregation_comparison_L8.csv


## 8. Compute polygon-interior s values — L8 sample, all epochs and variables

In [18]:
# Cell 10 — Compute s values — L8 sample, all epochs
def compute_s_values(sample_ids, geom_gdf, hyde_data, lons, lats, label=''):
    """
    Polygon-interior mean for each sample basin, all variables and epochs.
    Returns DataFrame: index=hybas_id, columns=(var, year).
    """
    rows = []
    t0 = time.time()
    n = sum(1 for hid in sample_ids if hid in geom_gdf.index)
    done = 0
    
    for hid in sample_ids:
        if hid not in geom_gdf.index:
            continue
        poly = geom_gdf.loc[hid, 'geometry']
        row = {'hybas_id': hid}
        for vname in HYDE_VARS:
            for yr in TARGET_YEARS:
                val, _ = polygon_interior_mean(poly, hyde_data[vname][yr], lons, lats)
                row[f'{vname}_{yr}'] = val
        rows.append(row)
        done += 1
        if done % 500 == 0:
            elapsed = time.time() - t0
            print(f'  {label} {done}/{n} in {elapsed:.1f}s (est. total: {elapsed/done*n:.0f}s)')
    
    elapsed = time.time() - t0
    print(f'  {label} complete: {done} basins in {elapsed:.1f}s')
    return pd.DataFrame(rows).set_index('hybas_id')


# L8 sample
print('Computing s values for L8 sample...')
s8 = compute_s_values(sample8_ids, geom8, hyde_data, hyde_lons, hyde_lats, label='L8')
print(s8.describe())

  L8 500/500 in 1.9s (est. total: 2s)
  L8 complete: 500 basins in 1.9s
       cropland_-1000  cropland_0  cropland_1000  cropland_2000  \
count      500.000000  500.000000     500.000000     500.000000   
mean         0.403683    0.852143       1.103288       7.315365   
std          1.647228    3.145302       3.464943      13.386523   
min          0.000000    0.000000       0.000000       0.000000   
25%          0.000000    0.000000       0.000000       0.000000   
50%          0.000000    0.000000       0.000000       0.007983   
75%          0.000000    0.020695       0.146851       8.005197   
max         18.195492   25.035113      24.070939      69.290774   

       grazing_land_-1000  grazing_land_0  grazing_land_1000  \
count          500.000000      500.000000         500.000000   
mean             0.463290        0.794815           1.615668   
std              1.932994        2.807602           4.704078   
min              0.000000        0.000000           0.000000   
25% 

In [19]:
# Cell 11 — Compute s values — L6 sample, all epochs
print('Computing s values for L6 sample...')
s6 = compute_s_values(sample6_ids, geom6, hyde_data, hyde_lons, hyde_lats, label='L6')
print(s6.describe())

  L6 500/500 in 18.3s (est. total: 18s)
  L6 complete: 500 basins in 18.3s
       cropland_-1000    cropland_0  cropland_1000  cropland_2000  \
count      500.000000  5.000000e+02     500.000000     500.000000   
mean         0.452126  9.822532e-01       1.235038       8.456220   
std          2.094106  3.724419e+00       3.846832      13.778185   
min          0.000000  0.000000e+00       0.000000       0.000000   
25%          0.000000  0.000000e+00       0.000000       0.000000   
50%          0.000000  1.317270e-07       0.022365       1.054733   
75%          0.053059  1.982822e-01       0.434047      12.260901   
max         28.462831  4.706543e+01      46.016880      74.868117   

       grazing_land_-1000  grazing_land_0  grazing_land_1000  \
count          500.000000      500.000000         500.000000   
mean             0.549152        0.917019           2.114046   
std              1.827381        2.551031           5.045967   
min              0.000000        0.000000      

## 9. Compute upstream u values — centroid lookup + sub_area weighting

In [20]:
# Cell 12 — Compute u values — L8 sample
def compute_u_values(sample_ids, topo_df, dag, hyde_data, lons, lats, label=''):
    """
    Upstream area-weighted mean HYDE value for each sample basin.
    Uses centroid lookup for upstream basins (fast, vectorized per basin).
    Returns DataFrame: index=hybas_id, columns=(var, year).
    """
    # Pre-compute centroid arrays from topo_df for fast lookup
    topo_lons = topo_df['lon'].values
    topo_lats = topo_df['lat'].values
    topo_ids = topo_df.index.values
    id_to_idx = {hid: i for i, hid in enumerate(topo_ids)}
    
    rows = []
    t0 = time.time()
    
    for i, hid in enumerate(sample_ids):
        if hid not in topo_df.index:
            continue
        
        upstream_ids = get_all_upstream(hid, dag)
        
        if not upstream_ids:
            # Headwater: u = s (no upstream)
            row = {'hybas_id': hid, 'n_upstream': 0}
            for vname in HYDE_VARS:
                for yr in TARGET_YEARS:
                    row[f'{vname}_{yr}'] = np.nan  # will fill from s later
            rows.append(row)
            continue
        
        # Get sub_areas for upstream basins present in topo_df
        valid_upstream = [uid for uid in upstream_ids if uid in topo_df.index]
        if not valid_upstream:
            rows.append({'hybas_id': hid, 'n_upstream': 0})
            continue
        
        up_df = topo_df.loc[valid_upstream]
        up_areas = up_df['sub_area'].values
        total_up_area = up_areas.sum()
        
        up_lons = up_df['lon'].values
        up_lats = up_df['lat'].values
        
        row = {'hybas_id': hid, 'n_upstream': len(valid_upstream), 'total_up_area': total_up_area}
        
        for vname in HYDE_VARS:
            for yr in TARGET_YEARS:
                # Vectorized centroid lookup for all upstream basins
                vals = batch_centroid_lookup(up_lons, up_lats, hyde_data[vname][yr], lons, lats)
                # Area-weighted mean
                u_val = np.average(vals, weights=up_areas)
                row[f'{vname}_{yr}'] = float(u_val)
        
        rows.append(row)
        
        if (i + 1) % 100 == 0:
            elapsed = time.time() - t0
            total = sum(1 for sid in sample_ids if sid in topo_df.index)
            print(f'  {label} {i+1}/{total} in {elapsed:.1f}s')
    
    print(f'  {label} u complete in {time.time()-t0:.1f}s')
    return pd.DataFrame(rows).set_index('hybas_id')


print('Computing u values for L8 sample...')
u8 = compute_u_values(sample8_ids, topo8, dag8, hyde_data, hyde_lons, hyde_lats, label='L8')
print(u8[['n_upstream', 'total_up_area']].describe())

  L8 500/500 in 11.3s
  L8 u complete in 11.3s
        n_upstream  total_up_area
count   500.000000   2.020000e+02
mean    159.014000   2.810829e+05
std     692.899341   7.770424e+05
min       0.000000   4.086000e+02
25%       0.000000   2.011825e+03
50%       0.000000   5.282000e+03
75%       6.000000   5.293128e+04
max    6093.000000   5.289732e+06


In [21]:
# Cell 13 — Compute u values — L6 sample
print('Computing u values for L6 sample...')
u6 = compute_u_values(sample6_ids, topo6, dag6, hyde_data, hyde_lons, hyde_lats, label='L6')
print(u6[['n_upstream', 'total_up_area']].describe())

  L6 u complete in 0.7s
       n_upstream  total_up_area
count  500.000000   1.400000e+02
mean     8.926000   3.037880e+05
std     36.773484   6.977900e+05
min      0.000000   5.406100e+03
25%      0.000000   2.869130e+04
50%      0.000000   5.550140e+04
75%      4.000000   1.977783e+05
max    358.000000   5.393047e+06


## 10. s/u divergence distributions

In [22]:
# Cell 14 — Divergence summary table
def compute_divergence(s_df, u_df, var, yr):
    """
    log2(u/s) divergence for basins with both s and u values.
    Returns series indexed by hybas_id.
    """
    col = f'{var}_{yr}'
    common = s_df.index.intersection(u_df.index)
    s_vals = s_df.loc[common, col]
    u_vals = u_df.loc[common, col]
    # Only compute where both s and u are meaningful (> 0.001 km²)
    mask = (s_vals > 0.001) & (u_vals > 0.001)
    div = np.log2(u_vals[mask] / s_vals[mask])
    return div, mask.sum(), len(common)


# Summary table
print('s/u divergence summary (log2(u/s), computed only where both s,u > 0.001 km²):')
print(f'{"Level":<5} {"Variable":<14} {"Epoch":<10} {"N":<6} {"p05":<8} {"median":<8} {"p95":<8} {"p99":<8}')
print('-' * 65)

div_records = []
for level, s_df, u_df in [('L8', s8, u8), ('L6', s6, u6)]:
    for var in HYDE_VARS:
        for yr in TARGET_YEARS:
            div, n_pairs, n_common = compute_divergence(s_df, u_df, var, yr)
            if len(div) < 5:
                continue
            rec = {
                'level': level, 'variable': var, 'year': yr,
                'n_pairs': n_pairs, 'n_common': n_common,
                'p05': np.percentile(div, 5),
                'p25': np.percentile(div, 25),
                'median': np.percentile(div, 50),
                'p75': np.percentile(div, 75),
                'p95': np.percentile(div, 95),
                'p99': np.percentile(div, 99) if len(div) > 100 else np.nan,
            }
            div_records.append(rec)
            print(f'{level:<5} {var:<14} {EPOCH_LABELS[yr]:<10} {n_pairs:<6} '
                  f'{rec["p05"]:>7.2f}  {rec["median"]:>7.2f}  {rec["p95"]:>7.2f}  '
                  f'{rec["p99"]:>7.2f}')

div_summary = pd.DataFrame(div_records)
div_summary.to_csv(OUT_DIR + '09_su_divergence_summary.csv', index=False)
print('\nSaved 09_su_divergence_summary.csv')

s/u divergence summary (log2(u/s), computed only where both s,u > 0.001 km²):
Level Variable       Epoch      N      p05      median   p95      p99     
-----------------------------------------------------------------
L8    cropland       1000 BCE   42       -4.05    -0.91     2.83      nan
L8    cropland       1 CE       52       -3.49    -0.37     2.94      nan
L8    cropland       1000 CE    65       -3.39    -0.13     3.36      nan
L8    cropland       2000 CE    92       -3.78    -0.15     2.34      nan
L8    grazing_land   1000 BCE   50       -1.47     0.25     3.01      nan
L8    grazing_land   1 CE       61       -1.50     0.13     4.33      nan
L8    grazing_land   1000 CE    64       -1.23     0.02     3.85      nan
L8    grazing_land   2000 CE    104      -1.33     0.04     1.91     7.15
L6    cropland       1000 BCE   32       -4.49    -1.00     3.00      nan
L6    cropland       1 CE       44       -5.00    -0.59     2.11      nan
L6    cropland       1000 CE    62       

In [23]:
# Cell 15 — Plot divergence distributions: L8 vs L6, cropland vs grazing, across epochs
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey='row')

for row_i, (level, s_df, u_df) in enumerate([('L8', s8, u8), ('L6', s6, u6)]):
    for col_i, yr in enumerate(TARGET_YEARS):
        ax = axes[row_i, col_i]
        for var, color in [('cropland', '#2196F3'), ('grazing_land', '#4CAF50')]:
            div, _, _ = compute_divergence(s_df, u_df, var, yr)
            if len(div) > 2:
                ax.hist(div.clip(-6, 6), bins=30, alpha=0.5, color=color, label=var, density=True)
        ax.axvline(0, color='black', lw=0.8, ls='--')
        ax.set_title(f'{level} — {EPOCH_LABELS[yr]}')
        ax.set_xlabel('log₂(u/s)')
        if col_i == 0:
            ax.set_ylabel('Density')
        if row_i == 0 and col_i == 3:
            ax.legend(fontsize=8)

fig.suptitle('HYDE s/u divergence: log₂(upstream/local), by level and epoch', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR + '09_su_divergence_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 09_su_divergence_distributions.png')

Saved 09_su_divergence_distributions.png


## 11. Cross-check: HYDE 2000 CE vs static BasinATLAS cropland

In [24]:
# Cell 16 — Compare HYDE 2000 CE polygon-interior cropland to static crp_pc_sse / crp_pc_use
# HYDE units: km² of cropland in cell
# crp_pc_sse: percent of basin area under cropland (0–100)
# Convert HYDE to fraction: sum(cropland_km2 over cells in basin) / sub_area * 100
# BUT polygon_interior_mean returns mean km² per cell, not total km²
# To get pct: need total cropland km² = mean_per_cell * n_cells_in_basin * cell_area_km2
# Cell area at lat φ: (cell_deg * 111.32)^2 * cos(φ) km²
# ... or more simply: HYDE cropland values ARE in km² per cell, so mean over cells = mean intensity,
# not total extent. For comparison we need total / sub_area.
# For now, compare as correlation (rank) rather than absolute equality.

# L8: merge s8 2000 CE cropland with static crp_pc_sse
xcheck8 = s8[['cropland_2000']].join(topo8[['sub_area', 'crp_pc_sse', 'crp_pc_use']], how='inner')
xcheck8 = xcheck8.dropna()
xcheck8 = xcheck8[xcheck8['crp_pc_sse'] > 0]  # only where static shows any cropland

# L6: same
xcheck6 = s6[['cropland_2000']].join(topo6[['sub_area', 'crp_pc_sse', 'crp_pc_use']], how='inner')
xcheck6 = xcheck6.dropna()
xcheck6 = xcheck6[xcheck6['crp_pc_sse'] > 0]

from scipy.stats import spearmanr, pearsonr

for label, xdf in [('L8', xcheck8), ('L6', xcheck6)]:
    r_s, p_s = spearmanr(xdf['cropland_2000'], xdf['crp_pc_sse'])
    r_p, _ = pearsonr(np.log1p(xdf['cropland_2000']), np.log1p(xdf['crp_pc_sse']))
    print(f'{label}: n={len(xdf)}, Spearman r={r_s:.3f} (p={p_s:.2e}), Pearson log-log r={r_p:.3f}')

L8: n=227, Spearman r=0.730 (p=4.49e-39), Pearson log-log r=0.707
L6: n=295, Spearman r=0.763 (p=1.82e-57), Pearson log-log r=0.747


In [25]:
# Cell 17
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (label, xdf) in zip(axes, [('L8 sample', xcheck8), ('L6 full', xcheck6)]):
    ax.scatter(
        np.log1p(xdf['crp_pc_sse']),
        np.log1p(xdf['cropland_2000']),
        alpha=0.3, s=8
    )
    ax.set_xlabel('log(1 + crp_pc_sse) — BasinATLAS static %')
    ax.set_ylabel('log(1 + HYDE 2000 CE) — polygon-interior km²/cell')
    ax.set_title(f'{label}: HYDE 2000 CE vs static cropland')

plt.tight_layout()
plt.savefig(OUT_DIR + '09_cropland_crosscheck.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 09_cropland_crosscheck.png')

Saved 09_cropland_crosscheck.png


## 12. Reference sites: Timbuktu, Ur, Kaifeng — s/u by epoch

In [34]:
# Cell 18 — Reference sites: s values by epoch + static vs HYDE comparison
import geopandas as gpd

REF_SITES = {
    'Timbuktu': 1080561810.0,
    'Ur':       2080818060.0,
    'Kaifeng':  4080602410.0,
}

# Fetch reference site geometries from DB
conn = db_connect()
ids_str = ','.join(str(int(i)) for i in REF_SITES.values())
ref_geom = gpd.read_postgis(
    f"SELECT hybas_id, geom FROM public.basin08 WHERE hybas_id IN ({ids_str})",
    conn, geom_col='geom'
).rename_geometry('geometry').set_index('hybas_id')
conn.close()
print('Reference geometries loaded:', list(ref_geom.index))

# --- Local (s) values by epoch (mean km² per HYDE cell × n_cells = total km²) ---
print('\n=== Local HYDE cropland — total km² in basin, then % of basin area ===')
print(f'{"Site":<12} {"sub_area km²":<16}', '  '.join(f'{EPOCH_LABELS[yr]+" km²":<14}' for yr in TARGET_YEARS))
print('-' * 90)

for site, hid in REF_SITES.items():
    try:
        poly = ref_geom.loc[hid, 'geometry']
        sub_area_km2 = topo8.loc[hid, 'sub_area']
        vals_km2 = []
        for yr in TARGET_YEARS:
            mean_v, n_cells = polygon_interior_mean(poly, hyde_data['cropland'][yr], hyde_lons, hyde_lats)
            vals_km2.append(mean_v * n_cells)
        print(f'{site:<12} {sub_area_km2:<16.1f}', '  '.join(f'{v:<14.2f}' for v in vals_km2))
    except Exception as e:
        print(f'{site}: ERROR — {e}')

# --- Static vs HYDE cropland (% of basin area) ---
print('\n=== Static BasinATLAS cropland vs HYDE cropland (% of basin area) ===')
print(f'{"Site":<12} {"static %":<12}', '  '.join(f'{EPOCH_LABELS[yr]:<10}' for yr in TARGET_YEARS))
print('-' * 72)

rows = []
for site, hid in REF_SITES.items():
    try:
        static_pct = topo8.loc[hid, 'crp_pc_sse']
        sub_area_km2 = topo8.loc[hid, 'sub_area']
        poly = ref_geom.loc[hid, 'geometry']
        hyde_pcts = []
        for yr in TARGET_YEARS:
            mean_v, n_cells = polygon_interior_mean(poly, hyde_data['cropland'][yr], hyde_lons, hyde_lats)
            total_km2 = mean_v * n_cells
            hyde_pcts.append(total_km2 / sub_area_km2 * 100 if sub_area_km2 > 0 else np.nan)
        print(f'{site:<12} {static_pct:<12.2f}', '  '.join(f'{p:<10.2f}' for p in hyde_pcts))
        rows.append({'site': site, 'hybas_id': int(hid), 'static_crp_pct': static_pct,
                     **{f'hyde_{EPOCH_LABELS[yr]}': p for yr, p in zip(TARGET_YEARS, hyde_pcts)}})
    except Exception as e:
        print(f'{site}: ERROR — {e}')

out = pd.DataFrame(rows)
out_path = '/output/edop/explore/09_ref_site_static_vs_hyde.csv'
out.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')
print(out.to_string(index=False))


/Users/karlg/envs/_edop/lib/python3.12/site-packages/geopandas/io/sql.py:467: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(spatial_ref_sys_sql, con)


## 13. Save complete s/u tables

In [35]:
# Cell 19 — Merge s and u for output
out8 = s8.add_prefix('s_').join(u8.add_prefix('u_'), how='outer')
out8 = out8.join(topo8[['sub_area', 'endo', 'crp_pc_sse', 'crp_pc_use']], how='left')
out8.to_csv(OUT_DIR + '09_hyde_su_L8.csv')
print(f'Saved 09_hyde_su_L8.csv: {len(out8)} rows, {out8.shape[1]} columns')

out6 = s6.add_prefix('s_').join(u6.add_prefix('u_'), how='outer')
out6 = out6.join(topo6[['sub_area', 'endo', 'crp_pc_sse', 'crp_pc_use']], how='left')
out6.to_csv(OUT_DIR + '09_hyde_su_L6.csv')
print(f'Saved 09_hyde_su_L6.csv: {len(out6)} rows, {out6.shape[1]} columns')

print('\nTask 9 complete.')

Saved 09_hyde_su_L8.csv: 500 rows, 22 columns
Saved 09_hyde_su_L6.csv: 500 rows, 22 columns

Task 9 complete.
